# Seaborn Study Guide

Statistical visualization built on Matplotlib. Run cells in Jupyter to see plots.

## 1. 1. Setup & Themes

Seaborn works with pandas DataFrames. Set a theme once at the top of your notebook and all plots inherit consistent styling.

**sns.set_theme and built-in datasets**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set global theme — do this once at the top
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# Seaborn ships with practice datasets
tips  = sns.load_dataset('tips')
iris  = sns.load_dataset('iris')
titanic = sns.load_dataset('titanic')

print(tips.head())
print(f"\ntips shape: {tips.shape}")
print(tips.dtypes)

**Available styles and palettes**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

styles   = ['darkgrid','whitegrid','dark','white','ticks']
palettes = ['deep','muted','pastel','bright','dark','colorblind']

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, style in zip(axes[0], styles[:3]):
    with sns.axes_style(style):
        ax2 = ax.twinx()   # just show colors
    sns.barplot(x=['A','B','C'], y=[3,5,4], ax=ax, palette='deep')
    ax.set_title(f'style={style}')

for ax, pal in zip(axes[1], palettes[:3]):
    sns.barplot(x=['A','B','C'], y=[3,5,4], ax=ax, palette=pal)
    ax.set_title(f'palette={pal}')

plt.tight_layout(); plt.show()

### Real-World: Setting Up a Project-Wide Visual Style

> A data team standardizes all report charts with a single style config at the top of every notebook.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Team-wide style configuration
sns.set_theme(
    style='whitegrid',
    palette='colorblind',   # accessible to color-blind viewers
    font_scale=1.0,
    rc={
        'figure.figsize':    (9, 4),
        'axes.spines.top':   False,
        'axes.spines.right': False,
        'grid.linewidth':    0.5,
    }
)

# Quick sanity-check plot
np.random.seed(42)
df = pd.DataFrame({
    'quarter': ['Q1','Q2','Q3','Q4'] * 3,
    'region':  ['North']*4 + ['South']*4 + ['East']*4,
    'revenue': np.random.uniform(100, 500, 12).round(1),
})

sns.barplot(data=df, x='quarter', y='revenue', hue='region')
plt.title('Revenue by Quarter & Region')
plt.ylabel('Revenue ($K)')
plt.tight_layout(); plt.show()

## 2. 2. Distribution Plots

histplot and kdeplot show how data is distributed. displot combines both into a faceted figure-level function.

**histplot and kdeplot**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histogram with KDE overlay
sns.histplot(data=tips, x='total_bill', bins=25,
             kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Total Bill Distribution')

# KDE only — compare two groups
sns.kdeplot(data=tips, x='tip', hue='sex',
            fill=True, alpha=0.4, ax=axes[1])
axes[1].set_title('Tip Distribution by Gender')

plt.tight_layout(); plt.show()

**displot — faceted distributions**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')

# Separate histogram per day, colored by sex
g = sns.displot(
    data=tips, x='total_bill', hue='sex',
    col='day', kind='hist', kde=True,
    bins=15, height=3.5, aspect=0.9,
    palette='Set2'
)
g.set_titles('{col_name}')
g.set_xlabels('Total Bill ($)')
plt.tight_layout(); plt.show()

### Real-World: Customer Spend Distribution by Segment

> A marketing analyst compares spend distributions across customer segments to set targeted promotional thresholds.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

np.random.seed(42)
segments = {
    'VIP':       np.random.normal(350, 80,  200),
    'Regular':   np.random.normal(150, 50,  500),
    'Occasional':np.random.normal(60,  30,  300),
}

rows = []
for seg, vals in segments.items():
    for v in vals.clip(0):
        rows.append({'segment': seg, 'spend': round(v, 2)})
df = pd.DataFrame(rows)

sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(10, 4))
sns.kdeplot(data=df, x='spend', hue='segment',
            fill=True, alpha=0.3, linewidth=2,
            palette={'VIP':'#e74c3c','Regular':'#3498db','Occasional':'#2ecc71'})

# Threshold lines
for thresh, label in [(100,'Entry'), (250,'Mid'), (400,'Premium')]:
    ax.axvline(thresh, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.text(thresh+3, ax.get_ylim()[1]*0.9, label, fontsize=8, color='gray')

ax.set_xlabel('Monthly Spend ($)')
ax.set_title('Customer Spend Distribution by Segment')
plt.tight_layout(); plt.show()

## 3. 3. Categorical Plots — Bar & Count

barplot shows mean ± CI of a numeric variable by category. countplot shows frequency. Both accept hue for a third dimension.

**barplot and countplot**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Mean tip by day + confidence interval
sns.barplot(data=tips, x='day', y='tip',
            hue='sex', palette='Set2', ax=axes[0])
axes[0].set_title('Avg Tip by Day & Gender')
axes[0].set_ylabel('Mean Tip ($)')

# Count of meals per day
sns.countplot(data=tips, x='day', hue='time',
              palette='pastel', ax=axes[1])
axes[1].set_title('Meal Count by Day & Time')

plt.tight_layout(); plt.show()

**Horizontal bar with order**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')

# Compute mean tip per day — sort for readability
order = tips.groupby('day')['tip'].mean().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=tips, y='day', x='tip',
            order=order, palette='Blues_d', orient='h', ax=ax)
ax.set_xlabel('Average Tip ($)')
ax.set_title('Average Tip by Day of Week (sorted)')
plt.tight_layout(); plt.show()

### Real-World: Product Return Rate by Category

> An operations analyst visualizes return rates across product categories to prioritize quality improvements.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

np.random.seed(7)
categories = ['Electronics','Clothing','Books','Toys','Sports','Home','Beauty']
data = []
for cat in categories:
    base_rate = np.random.uniform(0.03, 0.18)
    for _ in range(200):
        data.append({
            'category': cat,
            'returned': int(np.random.random() < base_rate),
            'channel':  np.random.choice(['Online','In-store'], p=[0.7, 0.3]),
        })
df = pd.DataFrame(data)

order = (df.groupby('category')['returned']
           .mean()
           .sort_values(ascending=False)
           .index.tolist())

sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=df, x='category', y='returned',
            hue='channel', order=order,
            palette={'Online':'#3498db','In-store':'#e67e22'},
            capsize=0.08, ax=ax)

ax.axhline(df['returned'].mean(), color='red', linestyle='--',
           linewidth=1.5, label=f"Overall avg: {df['returned'].mean():.1%}")
ax.set_ylabel('Return Rate'); ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_xlabel(''); ax.set_title('Product Return Rate by Category & Channel')
ax.legend(); plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()

## 4. 4. Box Plot & Violin Plot

Box plots show the 5-number summary (min, Q1, median, Q3, max). Violin plots also show the distribution shape via KDE.

**boxplot and violinplot**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=tips, x='day', y='total_bill',
            hue='time', palette='Set3', ax=axes[0])
axes[0].set_title('Total Bill — Box Plot')

sns.violinplot(data=tips, x='day', y='total_bill',
               hue='time', split=True,
               palette='Set2', inner='quartile', ax=axes[1])
axes[1].set_title('Total Bill — Violin Plot')

plt.tight_layout(); plt.show()

**boxenplot and stripplot overlay**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')

fig, ax = plt.subplots(figsize=(8, 5))

# Letter-value plot (boxenplot) for larger datasets
sns.boxenplot(data=tips, x='day', y='total_bill',
              palette='muted', ax=ax)

# Overlay raw points
sns.stripplot(data=tips, x='day', y='total_bill',
              color='black', size=3, alpha=0.3, ax=ax, jitter=True)

ax.set_title('Total Bill Distribution per Day (boxen + strip)')
ax.set_ylabel('Total Bill ($)')
plt.tight_layout(); plt.show()

### Real-World: Employee Salary Distribution by Department

> An HR analyst compares salary spreads across departments to detect pay equity issues before a compensation review.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

np.random.seed(42)
depts = {
    'Engineering':  (120000, 25000, 80),
    'Sales':        (85000,  20000, 60),
    'Marketing':    (90000,  18000, 40),
    'HR':           (70000,  12000, 30),
    'Data Science': (130000, 22000, 50),
    'Operations':   (75000,  15000, 55),
}

rows = []
for dept, (mean, std, n) in depts.items():
    salaries = np.random.normal(mean, std, n).clip(50000, 200000)
    for s in salaries:
        rows.append({'dept': dept, 'salary': round(s, -2),
                     'level': np.random.choice(['Junior','Mid','Senior'],
                                               p=[0.3,0.45,0.25])})
df = pd.DataFrame(rows)

order = df.groupby('dept')['salary'].median().sort_values(ascending=False).index

sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(11, 5))
sns.violinplot(data=df, x='dept', y='salary', order=order,
               palette='muted', inner='quartile', ax=ax)
sns.stripplot(data=df, x='dept', y='salary', order=order,
              hue='level', palette='dark:black', size=2.5,
              alpha=0.35, dodge=False, ax=ax, legend=False)

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
ax.set_xlabel(''); ax.set_ylabel('Annual Salary')
ax.set_title('Salary Distribution by Department')
plt.xticks(rotation=15, ha='right')
plt.tight_layout(); plt.show()

## 5. 5. Scatter & Regression Plots

scatterplot visualizes two numeric variables. regplot / lmplot adds a regression line with confidence interval automatically.

**scatterplot with hue and size**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=tips, x='total_bill', y='tip',
    hue='time', size='size',
    sizes=(40, 200), alpha=0.7,
    palette='Set1', ax=ax
)
ax.set_title('Tip vs Total Bill (size = party size)')
ax.set_xlabel('Total Bill ($)'); ax.set_ylabel('Tip ($)')
plt.tight_layout(); plt.show()

**regplot and lmplot**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# regplot — single regression line
sns.regplot(data=tips, x='total_bill', y='tip',
            scatter_kws=dict(alpha=0.4, s=30),
            line_kws=dict(color='red', linewidth=2), ax=axes[0])
axes[0].set_title('regplot: Tip vs Bill')

# lmplot — regression per group (returns FacetGrid)
plt.close()   # close to avoid conflict
g = sns.lmplot(data=tips, x='total_bill', y='tip',
               hue='smoker', palette='Set1',
               scatter_kws=dict(alpha=0.4, s=25),
               height=4, aspect=1.4)
g.set_axis_labels('Total Bill ($)', 'Tip ($)')
g.figure.suptitle('lmplot: Tip vs Bill by Smoker', y=1.02)
plt.tight_layout(); plt.show()

### Real-World: Advertising Spend vs Revenue

> A marketing data scientist plots ad spend against revenue by channel to compare ROI and fit regression lines.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

np.random.seed(5)
channels = ['Search','Social','Display','Email']
rows = []
for ch in channels:
    n      = 50
    spend  = np.random.uniform(500, 10000, n)
    slope  = {'Search':4.5,'Social':3.0,'Display':1.8,'Email':6.0}[ch]
    rev    = spend * slope + np.random.randn(n) * spend * 0.3
    for s, r in zip(spend, rev):
        rows.append({'channel':ch,'spend':round(s,0),'revenue':round(r,0)})
df = pd.DataFrame(rows)

sns.set_theme(style='whitegrid')
g = sns.lmplot(
    data=df, x='spend', y='revenue', hue='channel',
    col='channel', col_wrap=2,
    scatter_kws=dict(alpha=0.5, s=25),
    height=3.5, aspect=1.2,
    palette='tab10'
)
g.set_axis_labels('Ad Spend ($)', 'Revenue ($)')
g.set_titles('{col_name}')
g.figure.suptitle('Ad Spend vs Revenue by Channel', y=1.02, fontsize=13)

# Print ROI per channel
print("Estimated ROI (revenue/spend):")
for ch in channels:
    sub = df[df.channel==ch]
    roi = sub.revenue.sum() / sub.spend.sum()
    print(f"  {ch:8s}: {roi:.2f}x")
plt.tight_layout(); plt.show()

## 6. 6. Heatmap

sns.heatmap renders a matrix as color intensities. Ideal for correlation matrices, confusion matrices, and pivot table results.

**Correlation heatmap with annotations**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

sns.set_theme(style='white')
df = sns.load_dataset('tips')

# Select numeric columns
corr = df[['total_bill','tip','size']].corr()

fig, ax = plt.subplots(figsize=(5, 4))
mask = [[False, True, True],   # show lower triangle only
        [False, False, True],
        [False, False, False]]
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, linewidths=0.5,
            square=True, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout(); plt.show()

**Pivot table heatmap**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

np.random.seed(9)
months = ['Jan','Feb','Mar','Apr','May','Jun']
regions= ['North','South','East','West']
data   = pd.DataFrame({
    'month':   np.tile(months, 4),
    'region':  np.repeat(regions, 6),
    'revenue': np.random.uniform(50, 200, 24).round(1)
})

pivot = data.pivot(index='region', columns='month', values='revenue')

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu',
            linewidths=0.4, ax=ax, cbar_kws=dict(label='Revenue ($K)'))
ax.set_title('Monthly Revenue by Region ($K)')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.show()

### Real-World: Churn Risk Feature Correlation

> A customer success team uses a heatmap to identify which features are most correlated with customer churn.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

np.random.seed(3)
n = 500
df = pd.DataFrame({
    'tenure_months':    np.random.exponential(24, n).clip(1, 72),
    'monthly_spend':    np.random.normal(120, 40, n).clip(20),
    'support_tickets':  np.random.poisson(2, n),
    'logins_per_week':  np.random.normal(5, 3, n).clip(0),
    'nps_score':        np.random.normal(7, 2, n).clip(1, 10),
    'churned':          np.random.binomial(1, 0.25, n),
})
# Add realistic correlations
df['churned'] = (
    (df['support_tickets'] > 4).astype(int) * 0.4 +
    (df['logins_per_week'] < 2).astype(int) * 0.3 +
    (df['nps_score'] < 5).astype(int) * 0.3 +
    np.random.rand(n) * 0.3
) > 0.5

corr = df.corr(numeric_only=True)

sns.set_theme(style='white')
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, square=True, linewidths=0.5,
    ax=ax, annot_kws=dict(size=9)
)
ax.set_title('Customer Feature Correlation (churn focus)', pad=12)
plt.tight_layout(); plt.show()

## 7. 7. Pair Plot

pairplot creates a grid of scatter plots and distributions for all numeric column pairs — fast exploratory data analysis.

**Basic pairplot with hue**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
iris = sns.load_dataset('iris')

g = sns.pairplot(
    iris, hue='species',
    palette='Set2',
    plot_kws=dict(alpha=0.5, s=25),
    diag_kind='kde'
)
g.figure.suptitle('Iris Dataset — Pair Plot', y=1.02)
plt.tight_layout(); plt.show()

**pairplot with regression and custom diag**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')
cols = ['total_bill', 'tip', 'size']

g = sns.pairplot(
    tips[cols + ['time']], hue='time',
    kind='reg',          # scatter + regression line
    diag_kind='hist',    # histogram on diagonal
    palette='Set1',
    plot_kws=dict(scatter_kws=dict(alpha=0.3, s=20),
                  line_kws=dict(linewidth=1.5))
)
g.figure.suptitle('Tips Dataset — Regression Pair Plot', y=1.02)
plt.tight_layout(); plt.show()

### Real-World: Financial Feature Exploration Before Modeling

> A quant analyst uses pairplot to explore relationships between financial metrics before selecting features for a predictive model.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200
pe    = np.random.lognormal(2.8, 0.5, n)
eps   = np.random.normal(5, 2, n).clip(0.5)
rev_g = np.random.normal(0.12, 0.08, n)
roe   = np.random.normal(0.15, 0.07, n).clip(0)

df = pd.DataFrame({
    'P/E Ratio':   pe.round(2),
    'EPS ($)':     eps.round(2),
    'Rev Growth':  rev_g.round(3),
    'ROE':         roe.round(3),
    'sector':      np.random.choice(['Tech','Finance','Healthcare','Energy'], n),
})

sns.set_theme(style='whitegrid', font_scale=0.85)
g = sns.pairplot(
    df, hue='sector',
    vars=['P/E Ratio','EPS ($)','Rev Growth','ROE'],
    palette='tab10',
    plot_kws=dict(alpha=0.4, s=20),
    diag_kind='kde'
)
g.figure.suptitle('Financial Metrics — Pairplot by Sector', y=1.02)
plt.tight_layout(); plt.show()

## 8. 8. FacetGrid

FacetGrid tiles the same plot across subsets of data defined by row, col, and hue — the most powerful Seaborn layout tool.

**FacetGrid with map**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')

g = sns.FacetGrid(tips, col='time', row='smoker',
                  height=3, aspect=1.2, margin_titles=True)
g.map_dataframe(sns.histplot, x='total_bill', bins=15, kde=True)
g.set_axis_labels('Total Bill ($)', 'Count')
g.set_titles(row_template='{row_name}', col_template='{col_name}')
g.figure.suptitle('Total Bill by Time & Smoker', y=1.03)
plt.tight_layout(); plt.show()

**catplot — figure-level categorical plot**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')

# catplot wraps barplot/boxplot/etc into a FacetGrid
g = sns.catplot(
    data=tips, x='day', y='tip',
    col='time', kind='box',
    palette='Set2',
    height=4, aspect=0.9
)
g.set_titles('{col_name}')
g.set_axis_labels('Day', 'Tip ($)')
g.figure.suptitle('Tip Distribution by Day & Meal Time', y=1.03)
plt.tight_layout(); plt.show()

### Real-World: Multi-Region Sales Performance Tiles

> A BI engineer uses FacetGrid to generate one scatter panel per sales region, colored by product category.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

np.random.seed(11)
regions    = ['North','South','East','West']
categories = ['Electronics','Clothing','Food','Sports']

rows = []
for region in regions:
    for cat in categories:
        n = 40
        spend   = np.random.uniform(100, 5000, n)
        margin  = np.random.uniform(0.05, 0.45, n)
        rows.extend({'region':region,'category':cat,
                     'spend':round(s,0),'margin':round(m,3)}
                    for s,m in zip(spend, margin))
df = pd.DataFrame(rows)

sns.set_theme(style='whitegrid', font_scale=0.85)
g = sns.FacetGrid(df, col='region', col_wrap=2,
                  height=3.5, aspect=1.3, sharey=True)
g.map_dataframe(sns.scatterplot, x='spend', y='margin',
                hue='category', palette='tab10', alpha=0.6, s=25)

g.add_legend(title='Category')
g.set_axis_labels('Spend ($)', 'Margin')
g.set_titles('{col_name} Region')
g.figure.suptitle('Spend vs Margin by Region & Category', y=1.03)
plt.tight_layout(); plt.show()

## 9. 9. Time Series & Line Plot

sns.lineplot handles time series naturally — it aggregates multiple observations per x-value and draws confidence intervals.

**lineplot with hue and CI**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

np.random.seed(5)
weeks   = list(range(1, 13))
regions = ['North','South','East']

rows = []
for region in regions:
    base = np.random.uniform(80, 120)
    trend= np.random.uniform(1, 5)
    for w in weeks:
        for _ in range(5):   # 5 reps per point → CI makes sense
            rows.append({
                'week':   w,
                'region': region,
                'sales':  base + trend * w + np.random.randn() * 15
            })
df = pd.DataFrame(rows)

sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=df, x='week', y='sales', hue='region',
             palette='Set2', linewidth=2.5, ax=ax)
ax.set_title('Weekly Sales by Region (with 95% CI)')
ax.set_xlabel('Week'); ax.set_ylabel('Sales ($K)')
plt.tight_layout(); plt.show()

**relplot — faceted time series**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

np.random.seed(9)
products = ['Widget','Gadget','Doohickey']
months   = pd.date_range('2024-01-01', periods=12, freq='MS')

rows = []
for prod in products:
    base = np.random.uniform(50, 200)
    vals = base + np.cumsum(np.random.randn(12) * 10)
    for m, v in zip(months, vals):
        rows.append({'month':m,'product':prod,'revenue':max(v,10)})
df = pd.DataFrame(rows)

g = sns.relplot(data=df, x='month', y='revenue',
                col='product', kind='line',
                height=3, aspect=1.3,
                marker='o', markersize=5)
g.set_titles('{col_name}')
for ax in g.axes.flat:
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b'))
    ax.tick_params(axis='x', rotation=45)
g.figure.suptitle('Monthly Revenue by Product', y=1.03)
plt.tight_layout(); plt.show()

### Real-World: Server Metrics Time Series Dashboard

> A DevOps engineer plots CPU and memory usage over 24 hours across multiple servers with confidence bands.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

np.random.seed(20)
hours   = np.arange(24)
servers = [f'srv-{i:02d}' for i in range(1, 6)]

rows = []
for srv in servers:
    cpu_base  = np.random.uniform(20, 60)
    mem_base  = np.random.uniform(40, 70)
    for h in hours:
        spike = 30 if 9 <= h <= 17 else 0   # business hours spike
        rows.append({
            'hour':   h,
            'server': srv,
            'cpu':    (cpu_base + spike + np.random.randn()*8).clip(5,100),
            'memory': (mem_base + spike*0.3 + np.random.randn()*5).clip(10,95),
        })
df = pd.DataFrame(rows)

sns.set_theme(style='darkgrid')
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

sns.lineplot(data=df, x='hour', y='cpu',    hue='server',
             palette='tab10', linewidth=1.5, alpha=0.7, ax=axes[0])
axes[0].set_title('CPU Usage % (24h)'); axes[0].set_xlabel('Hour')

sns.lineplot(data=df, x='hour', y='memory', hue='server',
             palette='tab10', linewidth=1.5, alpha=0.7, ax=axes[1])
axes[1].set_title('Memory Usage % (24h)'); axes[1].set_xlabel('Hour')
axes[1].get_legend().remove()

# Highlight business hours
for ax in axes:
    ax.axvspan(9, 17, alpha=0.07, color='yellow', label='Business hours')

plt.tight_layout(); plt.show()

## 10. 10. Customization & Matplotlib Integration

Seaborn returns Axes objects you can modify with any Matplotlib method. Combine both libraries for full control.

**Accessing and modifying the Axes**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sns.set_theme(style='whitegrid')
tips = sns.load_dataset('tips')

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=tips, x='day', y='total_bill', palette='pastel', ax=ax)

# Matplotlib customizations on top
ax.set_title('Total Bill by Day', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('Total Bill', fontsize=11)
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:.0f}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add median annotations
medians = tips.groupby('day')['total_bill'].median()
for i, (day, med) in enumerate(medians.items()):
    ax.text(i, med + 0.5, f'${med:.0f}', ha='center', fontsize=9, color='darkred')

plt.tight_layout(); plt.show()

**Combining multiple Seaborn plots**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

sns.set_theme(style='whitegrid')
iris = sns.load_dataset('iris')

fig, ax = plt.subplots(figsize=(7, 5))

# Layer 1: violin
sns.violinplot(data=iris, x='species', y='petal_length',
               palette='pastel', inner=None, ax=ax)
# Layer 2: box inside violin
sns.boxplot(data=iris, x='species', y='petal_length',
            width=0.15, fliersize=0,
            boxprops=dict(facecolor='white', zorder=2), ax=ax)
# Layer 3: points
sns.stripplot(data=iris, x='species', y='petal_length',
              color='black', size=2.5, alpha=0.4, jitter=True, ax=ax)

ax.set_title('Petal Length by Species')
ax.set_ylabel('Petal Length (cm)')
plt.tight_layout(); plt.show()

### Real-World: Publication-Ready A/B Test Results

> A product analyst creates a publication-ready figure comparing two experiment variants with statistical annotations.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy import stats

np.random.seed(42)
control  = np.random.normal(45.0, 12, 500)
variant  = np.random.normal(48.5, 11, 500)
df = pd.DataFrame({
    'group':      ['Control']*500 + ['Variant']*500,
    'revenue':    np.concatenate([control, variant]),
})

t_stat, p_val = stats.ttest_ind(control, variant)

sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: violin + box + strip
for g, df_g in df.groupby('group'):
    pass   # just structure
sns.violinplot(data=df, x='group', y='revenue',
               palette={'Control':'#3498db','Variant':'#e74c3c'},
               inner=None, ax=axes[0])
sns.boxplot(data=df, x='group', y='revenue',
            width=0.1, fliersize=0,
            boxprops=dict(facecolor='white', zorder=2), ax=axes[0])
axes[0].set_title('Revenue Distribution'); axes[0].set_ylabel('Revenue ($)')

# Right: mean + CI bar chart
summary = df.groupby('group')['revenue'].agg(['mean','sem']).reset_index()
colors  = ['#3498db','#e74c3c']
bars    = axes[1].bar(summary['group'], summary['mean'],
                      yerr=summary['sem']*1.96, capsize=6,
                      color=colors, alpha=0.8, edgecolor='white', linewidth=1.5)
for bar, (_, row) in zip(bars, summary.iterrows()):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f"${row['mean']:.1f}", ha='center', fontsize=10, fontweight='bold')

sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
axes[1].annotate(f'p={p_val:.4f} {sig}', xy=(0.5,0.95), xycoords='axes fraction',
                 ha='center', fontsize=10, color='darkgreen' if p_val<0.05 else 'gray')
axes[1].set_title('Mean Revenue ± 95% CI'); axes[1].set_ylabel('Mean Revenue ($)')

fig.suptitle('A/B Test Results — Revenue', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()